# 03. Gold

Monta uma linha por sexta-feira e dia previsto, só com o que se sabia. Depois testa se alguma informação do futuro entrou. O código está em `funcoes/atributos.py` e `funcoes/verificacoes.py`.

In [1]:
from funcoes.atributos import processar_gold
from funcoes.verificacoes import testar_gold

processar_gold()
testar_gold()

gold_t1: (705, 79); 2023-02-24 a 2026-10-30; com consumo: 636; no treino: 622
gold_t2: (705, 77); 2023-02-24 a 2026-10-30; com consumo: 631; no treino: 617
Estrutura: origens em sextas, antecedencias corretas, sem plano humano em t+2.
t1: 79 colunas sem infinitos ou constantes; mais de 50% vazias: ['tipo_sugerido', 'plano_nutricionista']
t2: 77 colunas sem infinitos ou constantes; mais de 50% vazias: ['tipo_sugerido', 'previsao_chuva_mm', 'previsao_chuva_manha_mm', 'previsao_chuva_acum3d_mm', 'previsao_temp_max']


  2023-03-31: atributos identicos


  2023-11-03: atributos identicos


  2024-09-13: atributos identicos


  2025-05-23: atributos identicos


  2026-03-06: atributos identicos


  2026-10-30: atributos identicos
Nenhum atributo mudou com o futuro embaralhado.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 80, "display.max_columns", 30, "display.width", 180, "display.max_colwidth", 70)
GOLD = Path("..") / "dados" / "gold"
t1 = pd.read_parquet(GOLD / "gold_t1.parquet")
t2 = pd.read_parquet(GOLD / "gold_t2.parquet")
dicionario = pd.read_csv(GOLD / "gold_dicionario_atributos.csv", sep=";", encoding="utf-8-sig")
print(f"t+1: {t1.shape[0]} linhas | t+2: {t2.shape[0]} linhas")
print(f"Emissões: {t1['origem'].nunique()} | de {t1['origem'].min().date()} a {t1['origem'].max().date()}")
dicionario

t+1: 705 linhas | t+2: 705 linhas
Emissões: 160 | de 2023-02-24 a 2026-10-30


,grupo,coluna,disponivel_em,descricao
0,chave,origem,t1 e t2,Sexta-feira de emissao
1,chave,data,t1 e t2,Dia previsto
2,alvo,consumo,resultado,Refeicoes registradas; vazio em datas futuras ou sem registro
3,alvo,alvo_relativo,resultado,consumo / nivel_base
4,controle,tratamento,resultado,"normal, previsivel_calendario, alto_mantido, imprevisivel, periodo..."
5,controle,usar_no_treino,resultado,"Falso para dias imprevisiveis, greve, fora do calendario e sem con..."
6,nivel,nivel_base,t1 e t2,Media entre nivel_10d e nivel_mesmo_dia
7,nivel,nivel_10d,t1 e t2,Mediana dos 10 dias normais anteriores a emissao
8,nivel,nivel_20d,t1 e t2,Mediana dos 20 dias normais anteriores a emissao
9,nivel,nivel_mesmo_dia,t1 e t2,Mediana das 4 ultimas ocorrencias normais do mesmo dia da semana


## 1. Exemplo de uma emissão

Na emissão de 16/10/2026, o último consumo usado é anterior à própria sexta-feira, e o nível recente vem só de dias normais.

In [3]:
t1[t1["origem"] == "2026-10-16"][["origem", "data", "antecedencia_dias", "consumo", "plano_nutricionista", "nivel_base", "nivel_10d", "nivel_mesmo_dia",
                                 "idade_informacao_dias", "dias_ate_fim_bloco", "proteina", "tratamento"]].round(1)

<celula>:2: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  "idade_informacao_dias", "dias_ate_fim_bloco", "proteina", "tratamento"]].round(1)


,origem,data,antecedencia_dias,consumo,plano_nutricionista,nivel_base,nivel_10d,nivel_mesmo_dia,idade_informacao_dias,dias_ate_fim_bloco,proteina,tratamento
691,2026-10-16,2026-10-19,3,224.0,140.0,205.8,201.0,210.5,7,15.0,bovina,normal
692,2026-10-16,2026-10-20,4,185.0,140.0,200.5,201.0,200.0,7,15.0,frango,normal
693,2026-10-16,2026-10-21,5,202.0,130.0,208.8,201.0,216.5,7,15.0,bovina,normal
694,2026-10-16,2026-10-22,6,193.0,130.0,203.5,201.0,206.0,7,15.0,frango,normal
695,2026-10-16,2026-10-23,7,149.0,120.0,194.2,201.0,187.5,7,15.0,suina,normal


## 2. O que entra no treino

Os mesmos dias são usados nas duas janelas; muda apenas o que se sabia em cada emissão.

In [4]:
pd.DataFrame({"t+1": t1["tratamento"].value_counts(), "t+2": t2["tratamento"].value_counts()}).fillna(0).astype(int)

,t+1,t+2
tratamento,,
normal,570,565
sem_consumo,69,74
previsivel_calendario,32,32
alto_mantido,20,20
imprevisivel,11,11
periodo_de_greve,3,3


In [5]:
for nome, g in (("t+1", t1), ("t+2", t2)):
    resumo = g[g["consumo"].notna()].groupby("ano").agg(dias=("data", "size"), no_treino=("usar_no_treino", "sum"))
    print(nome); print(resumo.to_string()); print()

t+1
      dias  no_treino
ano                  
2023   169        166
2024   142        139
2025   169        168
2026   156        149

t+2
      dias  no_treino
ano                  
2023   164        161
2024   142        139
2025   169        168
2026   156        149



## 3. Cobertura dos atributos

In [6]:
atributos = ["nivel_base", "tendencia_nivel", "dias_ate_fim_bloco", "proteina", "efeito_proteina", "efeito_preparo", "reserva_discente", "plano_nutricionista"]
com_consumo = t1[t1["consumo"].notna()]
(com_consumo.groupby("ano")[atributos].apply(lambda d: d.notna().mean()) * 100).round(1).T

ano,2023,2024,2025,2026
nivel_base,100.0,100.0,100.0,100.0
tendencia_nivel,100.0,100.0,100.0,100.0
dias_ate_fim_bloco,100.0,100.0,100.0,100.0
proteina,100.0,100.0,99.4,91.7
efeito_proteina,97.0,100.0,99.4,91.7
efeito_preparo,96.4,99.3,99.4,91.7
reserva_discente,100.0,84.5,90.5,0.0
plano_nutricionista,0.0,0.0,0.0,100.0


Os dias sem bloco letivo definido e as lacunas de cardápio ficam vazios: cada modelo decidirá como tratá-los dentro do próprio treino, sem preenchimento prévio com dados do futuro.

In [7]:
ida = t1[t1["data"] <= "2026-10-30"]["idade_informacao_dias"]
print("Idade da informação em t+1 (dias):", ida.describe(percentiles=[.5, .9, .99]).round(1).to_dict())
print("Datas futuras prontas para previsão:")
futuro = t1[t1["data"] > "2026-10-30"]
print(pd.concat([futuro, t2[t2["data"] > "2026-10-30"]])[["origem", "data", "horizonte", "nivel_base", "dias_ate_fim_bloco", "proteina"]].round(1).to_string(index=False))

Idade da informação em t+1 (dias): {'count': 701.0, 'mean': 3.4, 'std': 9.7, 'min': 1.0, '50%': 1.0, '90%': 4.0, '99%': 56.0, 'max': 70.0}
Datas futuras prontas para previsão:
    origem       data horizonte  nivel_base  dias_ate_fim_bloco proteina
2026-10-30 2026-11-03        t1       194.0                15.0      NaN
2026-10-30 2026-11-04        t1       200.5                15.0      NaN
2026-10-30 2026-11-05        t1       195.0                15.0      NaN
2026-10-30 2026-11-06        t1       190.0                15.0      NaN
2026-10-23 2026-11-03        t2       195.2                15.0      NaN
2026-10-23 2026-11-04        t2       202.5                15.0      NaN
2026-10-23 2026-11-05        t2       200.8                15.0      NaN
2026-10-23 2026-11-06        t2       192.0                15.0      NaN
2026-10-30 2026-11-09        t2       202.5                15.0      NaN
2026-10-30 2026-11-10        t2       194.0                15.0      NaN
2026-10-30 2026-11-11

<celula>:5: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(pd.concat([futuro, t2[t2["data"] > "2026-10-30"]])[["origem", "data", "horizonte", "nivel_base", "dias_ate_fim_bloco", "proteina"]].round(1).to_string(index=False))


## 4. Previsão do tempo disponível na emissão

As colunas `previsao_` trazem o que a previsão dizia naquela sexta-feira, recuperado dos arquivos da Open-Meteo. A tabela compara essa previsão com o clima realizado e mostra se a chuva acompanha o consumo. `consumo_vs_*` usa só dias que entram no treino.

In [8]:
def avaliar_previsao(g, nome, limite=15):
    d = g[g["previsao_chuva_mm"].notna() & g["oraculo_chuva_mm"].notna()]
    evento, alerta = d["oraculo_chuva_mm"] >= limite, d["previsao_chuva_mm"] >= limite
    treino = d[d["usar_no_treino"] & d["alvo_relativo"].notna()]
    return {
        "janela": nome, "dias": len(d), "primeira_data": d["data"].min().date(),
        "corr_previsao_vs_chuva_real": round(d["previsao_chuva_mm"].corr(d["oraculo_chuva_mm"]), 2),
        "dias_com_15mm": int(evento.sum()), "alertas": int(alerta.sum()), "alertas_certos": int((evento & alerta).sum()),
        "consumo_vs_chuva_real": round(treino["oraculo_chuva_mm"].corr(treino["alvo_relativo"]), 3),
        "consumo_vs_previsao": round(treino["previsao_chuva_mm"].corr(treino["alvo_relativo"]), 3),
    }

pd.DataFrame([avaliar_previsao(t1, "Preparo, desde 2024"), avaliar_previsao(t1[t1["data"] >= t2.loc[t2["previsao_chuva_mm"].notna(), "data"].min()], "Preparo, mesmas datas de compras"), avaliar_previsao(t2, "Compras")]).set_index("janela").T

janela,"Preparo, desde 2024","Preparo, mesmas datas de compras",Compras
dias,519,117,117
primeira_data,2024-02-05,2026-04-13,2026-04-13
corr_previsao_vs_chuva_real,0.55,0.56,0.15
dias_com_15mm,40,10,10
alertas,19,8,7
alertas_certos,10,3,1
consumo_vs_chuva_real,0.018,0.008,-0.005
consumo_vs_previsao,0.113,0.058,0.028


In [9]:
atipicos = pd.to_datetime(["2026-09-09", "2026-09-10", "2026-09-11", "2026-09-15", "2026-09-23"])
cols = ["origem", "data", "consumo", "oraculo_chuva_mm", "previsao_chuva_mm"]
a = t1[t1["data"].isin(atipicos)][cols].rename(columns={"previsao_chuva_mm": "previsto_t1"})
b = t2[t2["data"].isin(atipicos)][["data", "previsao_chuva_mm"]].rename(columns={"previsao_chuva_mm": "previsto_t2"})
a.merge(b, on="data").round(1)

<celula>:5: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  a.merge(b, on="data").round(1)


,origem,data,consumo,oraculo_chuva_mm,previsto_t1,previsto_t2
0,2026-09-04,2026-09-09,84.0,15.1,30.5,4.4
1,2026-09-04,2026-09-10,103.0,38.3,19.6,0.3
2,2026-09-04,2026-09-11,138.0,12.8,1.6,3.9
3,2026-09-11,2026-09-15,14.0,7.4,5.2,3.7
4,2026-09-18,2026-09-23,23.0,4.3,4.8,18.8


## 5. Limites conhecidos

- **Previsão do tempo:** em t+1 há histórico desde 2024. Em t+2 há poucas rodadas, porque o serviço guarda um período curto, e a previsão quase não acompanha a chuva real nessa antecedência. `coletar_previsoes_clima.py` precisa rodar toda semana para preservar as rodadas.
- **Clima realizado:** as colunas `oraculo_` não estavam disponíveis na emissão e ficam fora das previsões.
- **Cardápio em t+2:** a base não registra quando cada cardápio foi publicado. Por isso os modelos de compras serão avaliados com e sem esses atributos.
- **Calendário:** a Gold usa a versão atual, que pode ter sido ajustada depois de publicada.
- **Plano da nutricionista:** existe só em 2026, então os modelos não o aprendem a partir das reservas discentes. As duas colunas permanecem separadas.